In [2]:
# read the input datset
with open('shakesphere_input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print("length if dataset: ", len(text))

length if dataset:  1115394


In [4]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [5]:
uniqu_chars = sorted(set(text))
vocab_size = len(uniqu_chars)
print(len(uniqu_chars))
print(''.join(uniqu_chars))

65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [6]:
stoi = { ch:i for i,ch in enumerate(uniqu_chars)}
itos = { i:ch for i,ch in enumerate(uniqu_chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hello there!"))
print(decode(encode("hello there!")))


[46, 43, 50, 50, 53, 1, 58, 46, 43, 56, 43, 2]
hello there!


In [7]:
#encode entire data set and store it into a toch.tensor
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

c:\Users\susha\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\_subclasses\functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [8]:
# seprate text into train and test split
n = int(0.9*len(text))
train_data= data[:n]
val_data = data[n:]
if len(train_data) + len(val_data) == len(text):
    print("train and test split looks good!") 

train and test split looks good!


In [9]:
torch.manual_seed(1657)
batch_size = 4 #how many individual sequences we shall process in parallel
block_size = 8 # what is the max context len for predictions


def get_batch(split):
    # genrate a small batch of data of i/ps x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x,y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets')
print(yb.shape)
print(yb)

print('------')

for b in range(batch_size) :
    for t in range(block_size):
        context  = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} target: {target}")

inputs:
torch.Size([4, 8])
tensor([[ 5,  1, 47, 44,  1, 42, 43, 39],
        [ 0, 35, 46, 63,  1, 50, 43, 58],
        [42,  1, 45, 47, 60, 43,  1, 59],
        [49,  1, 46, 47, 51, 11,  0, 25]])
targets
torch.Size([4, 8])
tensor([[ 1, 47, 44,  1, 42, 43, 39, 58],
        [35, 46, 63,  1, 50, 43, 58,  1],
        [ 1, 45, 47, 60, 43,  1, 59, 57],
        [ 1, 46, 47, 51, 11,  0, 25, 39]])
------
when input is [5] target: 1
when input is [5, 1] target: 47
when input is [5, 1, 47] target: 44
when input is [5, 1, 47, 44] target: 1
when input is [5, 1, 47, 44, 1] target: 42
when input is [5, 1, 47, 44, 1, 42] target: 43
when input is [5, 1, 47, 44, 1, 42, 43] target: 39
when input is [5, 1, 47, 44, 1, 42, 43, 39] target: 58
when input is [0] target: 35
when input is [0, 35] target: 46
when input is [0, 35, 46] target: 63
when input is [0, 35, 46, 63] target: 1
when input is [0, 35, 46, 63, 1] target: 50
when input is [0, 35, 46, 63, 1, 50] target: 43
when input is [0, 35, 46, 63, 1, 50, 43

In [10]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1657)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx


m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape) # (batch_size, block_size, vocab_size)
print(loss)

print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.9125, grad_fn=<NllLossBackward0>)

PBaZ CAI?QcZHrChL3VOsMTMjUr:uOsSKsc'uk!,u
ntoKEFSB;ca?;Rk!fg3H?I3o$o3nohti!,L;LLClaB;F3fxcRF?Y!q,SnS


In [11]:
optimier = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [12]:
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimier.zero_grad(set_to_none=True)
    loss.backward()
    optimier.step()

print(f"loss {loss.item()}")

loss 2.446537733078003


In [13]:
print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))



Ans?
O, thiothersw! e shangonchayout t
Ino a ere n at ungs wD u be spe imy u ur bes, s culod oun ht


In [14]:
torch.manual_seed(1657)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
print(x.shape)

torch.Size([4, 8, 2])


In [15]:
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b, t] = torch.mean(xprev,0)


In [16]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x 
torch.allclose(xbow, xbow2)

print(xbow[0])
print(xbow2[0])

tensor([[ 0.4412,  0.4433],
        [-0.7153,  0.7837],
        [-0.4673,  0.1072],
        [-0.1456,  0.3312],
        [ 0.1088,  0.2356],
        [-0.1090,  0.2502],
        [-0.0953,  0.4539],
        [ 0.0150,  0.4868]])
tensor([[ 0.4412,  0.4433],
        [-0.7153,  0.7837],
        [-0.4673,  0.1072],
        [-0.1456,  0.3312],
        [ 0.1088,  0.2356],
        [-0.1090,  0.2502],
        [-0.0953,  0.4539],
        [ 0.0150,  0.4868]])


In [17]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 =  wei @ x
torch.allclose(xbow2, xbow3)


True

In [19]:
# self-attention
torch.manual_seed(1657)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

# single head of attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)   # (B, T, head_size)
q = query(x) # (B, T, head_size)
v = value(x) # (B, T, head_size)

wei = q @ k.transpose(-2, -1) # ((B, T, head_size) @ (B, head_size, T)) -> (B, T, T)

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
out = wei @ v
out.shape
out

tensor([[[ 7.6043e-01,  9.5206e-01, -3.3900e-01,  8.6434e-02,  3.1773e-02,
           3.4590e-02, -9.6316e-01,  4.3937e-01,  5.0860e-01,  4.1247e-01,
           7.1961e-02, -2.0127e-01,  8.4427e-01, -8.5134e-01, -7.3934e-01,
           7.1740e-01],
         [ 1.3472e+00,  4.8840e-01,  5.0084e-01, -2.3337e-01,  2.8296e-01,
           7.7100e-01, -1.2608e+00,  2.4220e-01, -4.9217e-01,  5.3450e-01,
           6.8800e-01, -3.1525e-01,  7.8935e-01,  2.5463e-01, -7.3861e-01,
           2.0437e-01],
         [ 8.3569e-01,  3.8996e-01, -2.2195e-01,  3.4593e-02,  9.1750e-02,
           2.3131e-01, -6.7107e-01,  2.6754e-01, -8.4910e-02,  3.4178e-01,
           2.8032e-01, -2.9880e-01,  4.8978e-01, -1.6697e-02, -4.4783e-01,
           6.2995e-01],
         [ 1.2831e+00,  3.4083e-01,  4.5406e-01, -2.9381e-01,  3.1716e-01,
           6.5014e-01, -1.0782e+00,  2.2786e-01, -4.1230e-01,  4.4662e-01,
           6.2073e-01, -3.0520e-01,  6.5297e-01,  3.7703e-01, -6.6443e-01,
           2.4135e-01],
    